### TP : Classification de texte avec un RNN

Dans ce TP, nous allons construire un modèle de réseau de neurones récurrent (RNN) pour effectuer une analyse de sentiments sur le jeu de données IMDB. Nous suivrons les étapes suivantes :

1. Préparation de l'environnement et des données.
2. Création de l'encodeur de texte.
3. Construction et entraînement du modèle RNN.
4. Évaluation et visualisation des performances.

### Étape 1 : Préparation de l'environnement et des données

Importez les bibliothèques nécessaires (`numpy`, `tensorflow`, `tensorflow_datasets`, `matplotlib`).

In [1]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt

Chargez le jeu de données IMDB via `tensorflow_datasets` et séparez-le en ensembles d'entraînement et de test.

In [2]:
dataset, info = tfds.load('imdb_reviews', with_info=True, as_supervised=True)
train_dataset, test_dataset = dataset['train'], dataset['test']

Afficher un échantillon de données

In [3]:
for example, label in train_dataset.take(1):
  print('text: ', example.numpy())
  print('label: ', label.numpy())

text:  b"This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. Both are great actors, but this must simply be their worst role in history. Even their great acting could not redeem this movie's ridiculous storyline. This movie is an early nineties US propaganda piece. The most pathetic scenes were those when the Columbian rebels were making their cases for revolutions. Maria Conchita Alonso appeared phony, and her pseudo-love affair with Walken was nothing but a pathetic emotional plug in a movie that was devoid of any real meaning. I am disappointed that there are movies like this, ruining actor's like Christopher Walken's good name. I could barely sit through it."
label:  0


La cellule suivante prépare les ensembles de données en les mélangeant, les groupant en batchs, et en activant la prélecture pour optimiser les performances.

Pourquoi utiliser .prefetch ? <br>
Lors de l'entraînement d'un modèle, les étapes de traitement des données (chargement, déchiffrement, augmentation, etc.) peuvent devenir un goulot d'étranglement si elles prennent plus de temps que l'entraînement lui-même.<br>
Sans prefetch : Le modèle attend que les données soient préparées avant de continuer son entraînement.<br>
Avec prefetch : Les données pour la prochaine itération sont préparées en parallèle pendant que le modèle entraîne l'itération en cours.

Que fait tf.data.AUTOTUNE ?<br>
Le paramètre AUTOTUNE permet à TensorFlow de déterminer automatiquement le nombre optimal de lots de données à précharger dans le buffer, en fonction des ressources disponibles (CPU, GPU, mémoire).

In [5]:
BUFFER_SIZE = 10000
BATCH_SIZE = 64

train_dataset = train_dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Exemple d'affichage
for example, label in train_dataset.take(1):
    print('Exemples de textes :', example.numpy()[:3])
    print('Labels correspondants :', label.numpy()[:3])

Exemples de textes : [b'What\'s written on the poster is: "At birth he was given 6 years to live... At 34 he takes the journey of a lifetime." Ami is an American-born Israeli who was diagnosed with Muscular Dystrophy disease at the age of one. At age of 34, after the love toward his 22 years old care-giver didn\'t go well, he decided to come to the US to face the doctor who said that he would have only 6 years to live. He wanted to show the doctor that he is still alive, and weights 39 pounds. Why? Your guess is as good as mine, even I have seen this film.<br /><br />Obviously it\'s courageous to live when all he can move is his left index finger, but why does he have so much anger toward the doctor who diagnosed his disease 34 years ago? His doctor just told his mom that based on the medical history, people with his disease won\'t live long. What\'s the point of him showing up at old doctor\'s door for? Why is tracking down this old doctor in the US is a journey of his lifetime? There

### Étape 2 : Création de l'encodeur de texte

Utilisez `TextVectorization` de Keras pour encoder les textes en indices numériques. Adaptez l'encodeur aux données d'entraînement pour construire le vocabulaire. Note : dans TensorFlow Keras, fit est réservé à l'entraînement des modèles. Son pendant est adapt lorsqu'il concerne des couches qui "s'ajustent" aux données (prétraitement).


In [9]:
VOCAB_SIZE = 1000
encoder = tf.keras.layers.TextVectorization(max_tokens=VOCAB_SIZE)
encoder.adapt(train_dataset.map(lambda text, label: text))

Affichez les 20 premiers mots du vocabulaire pour validation.

In [10]:
# Affichage des 20 premiers mots du vocabulaire
vocab = np.array(encoder.get_vocabulary())
print("Premiers mots du vocabulaire :", vocab[:20])

Premiers mots du vocabulaire : ['' '[UNK]' 'the' 'and' 'a' 'of' 'to' 'is' 'in' 'it' 'i' 'this' 'that'
 'br' 'was' 'as' 'for' 'with' 'movie' 'but']


### Étape 3 : Construction du modèle RNN

Construisez un modèle séquentiel Keras composé des couches suivantes :
  1. L'encodeur de texte.
  2. Une couche d'embedding avec 64 dimensions.
  3. Une couche bidirectionnelle avec un LSTM de 64 unités.
  4. Une couche Dense avec 64 unités et activation ReLU.
  5. Une couche Dense avec une sortie unique (classification binaire).

In [ ]:
model = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(
        input_dim=len(encoder.get_vocabulary()),
        output_dim=64,
        mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1)
])

Compilez le modèle avec une perte de votre choix (bien choisie !), l'optimiseur `Adam` et la métrique `accuracy`.

In [12]:
model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(1e-4),
              metrics=['accuracy'])

Imaginons que nous n'ayons pas renseigné d'argument activation dans la dernière couche dense. Quel va être l'impact, s'il y en a un, par rapport au scénario où nous aurions renseigné "sigmoid" ? Proposer une règle pour convertir les outputs en prédictions binaires. 

### Étape 4 : Entraînement du modèle

Entraînez le modèle sur le jeu d'entraînement pour 10 epochs.

In [13]:
history = model.fit(train_dataset, epochs=1,
                    validation_data=test_dataset,
                    validation_steps=30)



391/391 [==============================] - 853s 2s/step - loss: 0.6485 - accuracy: 0.5490 - val_loss: 0.5330 - val_accuracy: 0.7323


### Etape 5 : Evaluation du modèle et affichage des résultats

Évaluez le modèle sur le jeu de test.
Affichez les courbes d'entraînement (précision et perte).

In [14]:
test_loss, test_acc = model.evaluate(test_dataset)
print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)

391/391 [==============================] - 109s 278ms/step - loss: 0.5266 - accuracy: 0.7452
Test Loss: 0.5266408324241638
Test Accuracy: 0.7451599836349487


In [15]:
def plot_graphs(history, metric):
    plt.plot(history.history[metric])
    plt.plot(history.history['val_' + metric], '')
    plt.xlabel("Époques")
    plt.ylabel(metric)
    plt.legend([metric, 'val_' + metric])

In [ ]:
# Affichage des courbes
plt.figure(figsize=(16, 8))
plt.subplot(1, 2, 1)
plot_graphs(history, 'accuracy')
plt.subplot(1, 2, 2)
plot_graphs(history, 'loss')

### Étape 6 : Prédiction sur de nouveaux textes

Effectuez une prédiction sur un texte d'exemple. Ajoutez un second texte rempli pour vérifier l'effet du padding sur les prédictions.

In [17]:
sample_text = ('The movie was cool. The animation and the graphics '
               'were out of this world. I would recommend this movie.')
padding = "the " * 2000

predictions = model.predict(np.array([sample_text, padding]))
print("Prédiction sur le texte :", predictions[0])

1/1 [==============================] - 2s 2s/step
Prédiction sur le texte : [0.19963379]


### Étape 7 : Ajouter une couche LSTM supplémentaire

Ajoutez une seconde couche bidirectionnelle LSTM (avec `return_sequences=True` sur la première). Réentraînez le modèle et observez les performances.

**A noter :** La sortie de la première couche LSTM est une séquence qui peut être traitée par la seconde couche.

In [ ]:
model = tf.keras.Sequential([
    encoder,
    tf.keras.layers.Embedding(len(encoder.get_vocabulary()), 64, mask_zero=True),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1)
])

model.compile(loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
              optimizer=tf.keras.optimizers.Adam(1e-4),
              metrics=['accuracy'])

history = model.fit(train_dataset, epochs=1,
                    validation_data=test_dataset,
                    validation_steps=30)

test_loss, test_acc = model.evaluate(test_dataset)
print('Test Loss:', test_loss)
print('Test Accuracy:', test_acc)